# 🌌 Stellar Class — 0.972 Weighted-Decorrelated Blend

**Playground Series S6E6** · metric = **balanced accuracy** · classes `GALAXY / QSO / STAR`.

This notebook reproduces a **top-region (~0.972)** submission by blending the strongest public
submissions with a **score-weighted, diversity-decorrelated vote**, then resolving the small
**disagreement region** with a probability consensus.

### How the leaderboard top is actually reached
Inspecting public submission histories (e.g. `zoli800`'s `submission_history.csv`) shows the very
top is reached by **coordinated leaderboard probing**: flip ~4 rows at a time
(`GALAXY↔STAR`, `GALAXY↔QSO`, `STAR↔GALAXY`), keep the change if the public score rises. The base
that probing starts from is a **blend of strong public models** — which is exactly what this
notebook builds.

### Add these datasets as inputs (Kaggle → Add Input)
- `nina2025/ps-s6e6` — pack of scored submissions (filenames = LB scores)
- `vladislavagamova/s6e6-097186-final-submission`
- `mehrankazeminia/s6e6-97181`
- `zoli800/s6e6-097209-final-submission` (optional, the current public best)
- `romonedunlop/s6e6-kaisei-error-features` (error-detector for the probing analysis)

> Full from-scratch pipeline (XGB/LGBM/Cat/HistGBM/MLP → TabPFN-3 meta) +
> AutoGluon variant: **https://github.com/MitudruDutta/Predicting-Stellar-Class**

In [ ]:
import numpy as np, pandas as pd, glob, os
from scipy.stats import mode
CLASSES = ['GALAXY','QSO','STAR']; C2I = {c:i for i,c in enumerate(CLASSES)}
COMP = '/kaggle/input/playground-series-s6e6'
test = pd.read_csv(f'{COMP}/test.csv'); test_ids = test['id'].values; M = len(test_ids)

def load_sub(path):
    d = pd.read_csv(path).sort_values('id').reset_index(drop=True)
    assert (d['id'].values == test_ids).all(), f'id mismatch: {path}'
    return d['class'].map(C2I).values

# ---- collect public submissions + parse their LB score from the filename ----
members = {}   # name -> (labels, lb_score)
def add(path, score, name=None):
    if os.path.exists(path):
        members[name or os.path.basename(path)] = (load_sub(path), score)

# nina pack: filename like 0.97183.csv  ->  score = float(stem)
for f in sorted(glob.glob('/kaggle/input/*/*.csv')):
    stem = os.path.splitext(os.path.basename(f))[0]
    try:
        sc = float(stem)
        if 0.96 < sc < 0.98:
            add(f, sc, name=f'nina_{stem}')
    except ValueError:
        pass
# explicitly-named strong subs
add('/kaggle/input/s6e6-097209-final-submission/submission.csv', 0.97209, 'zoli_209')
add('/kaggle/input/s6e6-097186-final-submission/submission.csv', 0.97186, 'slava_186')
add('/kaggle/input/s6e6-97181/submission.csv', 0.97181, 'kaze_181')

# keep only the strongest, de-duplicated, sorted by score
uniq = {}
for n,(lab,sc) in members.items():
    key = lab.tobytes()
    if key not in uniq or sc > uniq[key][2]:
        uniq[key] = (n, lab, sc)
members = {n:(lab,sc) for n,lab,sc in uniq.values()}
members = dict(sorted(members.items(), key=lambda kv:-kv[1][1])[:12])
print(f'{len(members)} unique strong members:')
for n,(_,sc) in members.items(): print(f'  {n}: {sc}')

## Weighted + decorrelated vote

Two ideas combined:
1. **Score weighting** — better submissions get more say (weight = score − min + ε).
2. **Decorrelation** — near-duplicate submissions are downweighted by their average agreement with
   the others, so a clique of identical files can't dominate the vote.

The blend can only change a row inside the **disagreement region** (where members differ), so we
also measure how small that region is.

In [ ]:
names = list(members)
L = np.stack([members[n][0] for n in names])          # (k, M)
S = np.array([members[n][1] for n in names])
k = len(names)

# disagreement region
unan = (L == L[0]).all(0)
print(f'unanimous rows: {unan.mean()*100:.2f}%  | disagreement region: {(~unan).sum()} rows')

# score weights
w = S - S.min() + 1e-4; w /= w.sum()

# decorrelation: uniqueness = 1 / mean agreement with the others
agr = np.array([[(L[a]==L[b]).mean() for b in range(k)] for a in range(k)])
off = agr.copy(); np.fill_diagonal(off, np.nan)
uniq = 1.0 / np.nanmean(off, 1); uniq /= uniq.mean()
wd = w * uniq; wd /= wd.sum()

# weighted vote
V = np.zeros((M, 3))
for j in range(k):
    np.add.at(V, (np.arange(M), L[j]), wd[j])
blend = V.argmax(1)
blend[unan] = L[0][unan]   # never override a unanimous vote
print('blend class dist:', {CLASSES[i]: int((blend==i).sum()) for i in range(3)})
if 'zoli_209' in names:
    print('agreement with best single (zoli_209):',
          round((blend==members['zoli_209'][0]).mean(), 5))

## Optional: probability-consensus tie-break on the disagreement region

Where members disagree, fall back to the **average class probability** if probability artifacts are
available (e.g. `cdeotte/s6e6-oof-and-test-preds`). Here we use the vote share itself as a soft
consensus — the larger the winning margin, the more confident the flip.

In [ ]:
# margin of the blended vote — high margin = confident
margin = V.max(1) - np.sort(V,1)[:,-2]
print('low-margin (contested) rows:', int((margin < 0.2).sum()))
# final = blend (already margin-aware via weighted vote). Anchor on best single where blend is unsure.
final = blend.copy()
if 'zoli_209' in names:
    anchor = members['zoli_209'][0]
    unsure = (~unan) & (margin < 0.15)
    final[unsure] = anchor[unsure]     # trust the strongest single submission on low-confidence rows
print('final vs blend flips:', int((final!=blend).sum()))

In [ ]:
sub = pd.DataFrame({'id': test_ids, 'class': np.array(CLASSES)[final]})
sub.to_csv('submission.csv', index=False)
print('submission.csv written', sub.shape)
print(sub['class'].value_counts()); sub.head()

## Notes & honest caveats

- This blend lands in the **~0.972 public region**; the exact public #1 is reached by additional
  **leaderboard probing** (4-row flips kept if the LB rises) on top of a blend like this. Probing
  optimises the *public* split specifically — guard against private-LB shuffle by also keeping a
  robust single model for final selection.
- **`class_weight='balanced'`** is the key lever for balanced accuracy when training your own bases.
- Residual error is dominated by **low-redshift GALAXY/STAR** photometric degeneracy.

⭐ From-scratch pipeline + TabPFN-3 meta + AutoGluon + the full probing ledger:
**https://github.com/MitudruDutta/Predicting-Stellar-Class**